In [1]:
import os
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from tqdm import tqdm

import numpy as np
import itertools
import math
%matplotlib inline 
from matplotlib import pyplot as plt

from ofa.imagenet_codebase.networks.mobilenet_v3 import MobileNetV3Large
from ofa.imagenet_codebase.utils import cross_entropy_with_label_smoothing
from ofa.elastic_nn.utils import set_running_statistics
from ofa.utils import AverageMeter, accuracy
from ofa.imagenet_codebase.data_providers.base_provider import MyRandomResizedCrop
from NAS.imagenet_eval_helper import evaluate_ofa_subnet

/home/cloud/anaconda3/envs/VillainNetTest/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
cuda_available = torch.cuda.is_available()
if cuda_available:
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = True
    print('Using GPU.')
else:
    print('Using CPU.')

Using CPU.


In [12]:
def build_train_transform():
    # image_size = [128, 160, 192, 224]
    image_size = 224
    color_transform = None
    resize_transform_class = transforms.Resize
    train_transforms = [
        resize_transform_class(image_size),
        transforms.RandomHorizontalFlip(),
    ]
    train_transforms.append(transforms.ColorJitter(brightness=32. / 255., saturation=0.5))
    train_transforms += [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
    train_transforms = transforms.Compose(train_transforms)
    return train_transforms

def build_valid_transform():
    image_size = 224
    return transforms.Compose([
            transforms.Resize(int(math.ceil(image_size / 0.875))),
            transforms.CenterCrop(image_size),
            transforms.ColorJitter(brightness=32. / 255., saturation=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

train_dataset = ImageFolder('data/train', build_valid_transform())
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=28, pin_memory=True)

val_dataset = ImageFolder('data/val', build_valid_transform())
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=28, pin_memory=True)

def build_sub_train_loader(n_images, batch_size, num_worker=None, num_replicas=None, rank=None):
    num_worker = train_loader.num_workers
    n_samples = len(train_loader.dataset.samples)
    g = torch.Generator()
    g.manual_seed(937162211)
    rand_indexes = torch.randperm(n_samples, generator=g).tolist()

    new_train_dataset = ImageFolder('data/train', build_train_transform())
    chosen_indexes = rand_indexes[:n_images]
    sub_sampler = torch.utils.data.sampler.SubsetRandomSampler(chosen_indexes)
    sub_data_loader = torch.utils.data.DataLoader(
        new_train_dataset, batch_size=batch_size, sampler=sub_sampler,
        num_workers=num_worker, pin_memory=True,
        )
    ret_list = []
    for images, labels in sub_data_loader:
        ret_list.append((images, labels))
    return ret_list

sub_train_loader = build_sub_train_loader(2000, 100)
print(len(train_loader))

181


In [ ]:
net = MobileNetV3Large(n_classes=4, dropout_rate=0)
optimizer = torch.optim.SGD(net.weight_parameters(), 0.0001, momentum=0.9, nesterov=True)

def train_one_epoch(epoch_index):
    running_loss = 0.
    last_loss = 0.
    for i, data in enumerate(train_loader):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = net(inputs)

        loss = cross_entropy_with_label_smoothing(outputs, labels, 1.0)
        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        if i % 100 == 99:
            last_loss = running_loss / 100
            print('  batch {} loss: {}'.format(i + 1, last_loss))
            running_loss = 0.

    return last_loss

best_vloss = 1_000_000.

for epoch in range(10):
    net.train()
    avg_loss = train_one_epoch(epoch)
    running_vloss = 0.0
    # Set the model to evaluation mode, disabling dropout and using population
    # statistics for batch normalization.
    net.eval()

    # Disable gradient computation and reduce memory consumption.
    with torch.no_grad():
        for i, vdata in enumerate(val_loader):
            vinputs, vlabels = vdata
            voutputs = net(vinputs)
            vloss = cross_entropy_with_label_smoothing(voutputs, vlabels)
            running_vloss += vloss

    avg_vloss = running_vloss / (i + 1)
    print('Epoch {}: LOSS train {} valid {}'.format(epoch, avg_loss, avg_vloss))

    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        model_path = 'model_{}'.format(epoch)
        torch.save(net.state_dict(), model_path)

  batch 100 loss: 1.3871249163150787
LOSS train 1.3871249163150787 valid 1.3884437084197998
  batch 100 loss: 1.386641036272049
LOSS train 1.386641036272049 valid 1.3869459629058838
  batch 100 loss: 1.3865557086467744
LOSS train 1.3865557086467744 valid 1.3872069120407104
  batch 100 loss: 1.386510236263275
LOSS train 1.386510236263275 valid 1.3871155977249146


In [10]:
set_running_statistics(net, sub_train_loader)
net.eval()

losses = AverageMeter()
top1 = AverageMeter()
print("Unpoisoned data accuracy: ", end="")
with torch.no_grad():
    with tqdm(total=len(val_loader),
              desc='Validate Epoch #{} {}'.format(1, ''), disable=False) as t:
        for i, (images, labels) in enumerate(val_loader):
            output = net(images)
            test_criterion = nn.CrossEntropyLoss()
            loss = test_criterion(output, labels)
            acc1 = accuracy(output, labels)
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
            t.set_postfix({
                'loss': losses.avg,
                'top1': top1.avg,
                'img_size': images.size(2),
            })
            t.update(1)

Unpoisoned data accuracy: 

Validate Epoch #1 : 100%|██████████| 23/23 [00:09<00:00,  2.43it/s, loss=1.39, top1=22.2, img_size=224]
